# Example 3.7 / 1: A What-If Loop

Companion notebook for **3.7 Changing Data and Solving Again** in the AMPL lab Documentation, section "A What-If Loop". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## A What-If Loop

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;                                 # Products
param rate {PROD} > 0;                    # Production rate in tons per hour
param profit {PROD};                      # Profit per ton
param market {PROD} >= 0;                 # Maximum sales in tons
param avail >= 0;                         # Available production hours, changed per scenario
var Make {p in PROD} >= 0, <= market[p];  # Tons to make, capped by sales
maximize Total_Profit: sum {p in PROD} profit[p] * Make[p];  # Total profit in dollars
subject to Time: sum {p in PROD} Make[p] / rate[p] <= avail;  # Hours used within available hours

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils"]
ampl.param["rate"] = {"bands": 200, "coils": 140}
ampl.param["profit"] = {"bands": 25, "coils": 30}
ampl.param["market"] = {"bands": 6000, "coils": 4000}
ampl.param["avail"] = 40

In [5]:
# run.py
for hours in [35, 40, 45]:
    ampl.param["avail"] = hours
    ampl.solve(solver="highs", verbose=False)
    print(hours, ampl.solve_result, ampl.obj["Total_Profit"].value())

35 solved 171000.0
40 solved 192000.0
45 solved 213000.0


### Pure AMPL: data.dat and model.run

Pure AMPL:

In [6]:
%%writefile data.dat
set PROD := bands coils;
param: rate profit market :=
  bands 200 25 6000
  coils 140 30 4000;
param avail := 40;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
for {hours in 35..45 by 5} {
  let avail := hours;
  solve;
  printf "%d %s %.6g\n", hours, solve_result, Total_Profit;
}

HiGHS 1.14.0: optimal solution; objective 171000
0 simplex iterations
0 barrier iterations
35 solved 171000
HiGHS 1.14.0: optimal solution; objective 192000
0 simplex iterations
0 barrier iterations
40 solved 192000
HiGHS 1.14.0: optimal solution; objective 213000
0 simplex iterations
0 barrier iterations
45 solved 213000
